# Four signals the fixes worked

Four signals below argue second version of dataset generation produced a more realistic dataset, not just a larger one. Every number is computed live against the current labels and the frozen pre-fix backup (`labels.backup-pre-idf-full.parquet`) — nothing here is pasted from a chat transcript.

**Prerequisites**: `data/route_labels/labels.parquet` and `data/route_labels/labels.backup-pre-idf-full.parquet` on disk.

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Setup

In [7]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from composition import CellFill
from composition.cells import CELLS
from hybrid_search_rrf_dataset.labels import RouteLabels
from hybrid_search_rrf_dataset.objective import RouterObjective
from hybrid_search_rrf_dataset.router import decisive_rows


def show(df: pd.DataFrame) -> None:
    display(Markdown(df.to_markdown(index=False)))


selection = CellFill().build()
labels = RouteLabels(selection, objective=RouterObjective(min_relevance=1))
current = labels.load()
before = pd.read_parquet(
    labels.labels_path.parent / "labels.backup-pre-idf-full.parquet"
)
print(f"current: {len(current):,} labelled rows | before: {len(before):,} labelled rows")

current: 46,142 labelled rows | before: 20,815 labelled rows


In [14]:
routes_differ = len(current[current['shape'] == 'routes_differ'])
all_tied = len(current[current['shape'] == 'all_tied'])

routes_differ, all_tied

(22943, 15018)

## Signal 1 — Doubled, Not Diluted

If the two fixes had mostly added noise, `labelled` would grow while `decisive` lagged behind — more rows, no more clean signal. Instead every stage of the funnel moved by roughly the same multiple: the extra volume carried the same *density* of trainable and decisive rows as what was already there, rather than diluting it.

`decisive` (from `decisive_rows`) is the strict subset of `routes_differ`: the winning route hit rank 1 *and* the runner-up missed entirely — `RouterObjective.decisive_margin` = hit_weight − ndcg_weight = 0.4. Most of the `routes_differ` → `decisive` drop is this deliberately strict bar ("both routes found something relevant" doesn't clear it), not an embedding weakness — see Signal 2.

In [15]:
def funnel(df: pd.DataFrame) -> dict:
    differ = df[df["shape"] == "routes_differ"]
    dec = decisive_rows(df)
    return {"labelled": len(df), "trainable": len(differ), "decisive": len(dec)}


f_before, f_after = funnel(before), funnel(current)
funnel_tbl = pd.DataFrame(
    [
        {
            "stage": stage,
            "before": f_before[stage],
            "after": f_after[stage],
            "multiple": round(f_after[stage] / f_before[stage], 2),
        }
        for stage in ("labelled", "trainable", "decisive")
    ]
)
show(funnel_tbl)

| stage     |   before |   after |   multiple |
|:----------|---------:|--------:|-----------:|
| labelled  |    20815 |   46142 |       2.22 |
| trainable |    11018 |   22943 |       2.08 |
| decisive  |     2607 |    5100 |       1.96 |

## Signal 2 — Balance Under Scrutiny

`decisive` rows are the hard-to-fake subset from Signal 1 — no thin margins, no ties to hide behind. If the trainable route mix were an artifact of noise (e.g. sparse winning mostly on coin-flip margins the IDF fix just happened to tip), the decisive mix would look different from the trainable mix. It doesn't — the two are within ~1.6 points of each other on every route, at a 4.5x smaller sample. The balance survives the strictest filter the dataset has.

In [16]:
def route_mix(df: pd.DataFrame, col: str) -> pd.Series:
    vc = df[col].value_counts()
    return (vc / vc.sum() * 100).round(1)


differ_now = current[current["shape"] == "routes_differ"]
dec_now = decisive_rows(current)

mix_tbl = pd.DataFrame(
    {
        "trainable %": route_mix(differ_now, "route"),
        "decisive %": route_mix(dec_now, "winner"),
    }
)
mix_tbl["gap_pts"] = (mix_tbl["trainable %"] - mix_tbl["decisive %"]).abs().round(1)
show(mix_tbl.reset_index(names="route"))

| route       |   trainable % |   decisive % |   gap_pts |
|:------------|--------------:|-------------:|----------:|
| dense_only  |          60.2 |         61.7 |       1.5 |
| sparse_only |          33.9 |         33.9 |       0   |
| pure_rrf    |           5.9 |          4.3 |       1.6 |

## Signal 3 — The Hedge Earns Its Keep

`headroom_decomposition` (SPEC d44a) names the single best blind route — the one line of code a router has to beat. RRF's entire design promise is "never much worse than the better of dense/sparse"; that hedge only pays off as a *blind default* once sparse is a real enough contender to be worth hedging against. Before the fixes, sparse was starved (TF-only scoring, ~8% of trainable rows) and blanket-dense won outright with nothing to hedge. After, the best global constant flips to `pure_rrf`.

The visible ceiling over that best constant *shrinks* as a direct consequence — the naive blind baseline got smarter, so there's less headroom left for a router to claim. That's the dataset getting more honest, not worse.

In [17]:
def scored(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    score_cols = [c for c in df.columns if c.startswith("score_")]
    ordered = np.sort(df[score_cols].to_numpy(), axis=1)
    return df.assign(oracle=ordered[:, -1]), score_cols


def headroom_decomposition_of(df: pd.DataFrame) -> pd.DataFrame:
    scored_df, score_cols = scored(df)
    global_constant = max(scored_df[c].mean() for c in score_cols)
    global_name = max(score_cols, key=lambda c: scored_df[c].mean())
    lane_best = scored_df.groupby("dataset")[score_cols].mean().max(axis=1)
    lane_n = scored_df.groupby("dataset").size()
    per_collection = float((lane_best * lane_n).sum() / lane_n.sum())
    oracle = float(scored_df["oracle"].mean())
    levels = pd.DataFrame(
        {
            "level": [
                f"one global constant ({global_name.removeprefix('score_')})",
                "best constant per collection",
                "per-query oracle (ceiling)",
            ],
            "score": [float(global_constant), per_collection, oracle],
        }
    )
    levels["gain_vs_previous_pct"] = levels["score"].pct_change().mul(100).round(1)
    return levels


print("BEFORE:")
show(headroom_decomposition_of(before).round(3))
print("AFTER:")
show(headroom_decomposition_of(current).round(3))

BEFORE:


| level                            |   score |   gain_vs_previous_pct |
|:---------------------------------|--------:|-----------------------:|
| one global constant (dense_only) |   0.404 |                    nan |
| best constant per collection     |   0.437 |                      8 |
| per-query oracle (ceiling)       |   0.494 |                     13 |

AFTER:


| level                          |   score |   gain_vs_previous_pct |
|:-------------------------------|--------:|-----------------------:|
| one global constant (pure_rrf) |   0.567 |                  nan   |
| best constant per collection   |   0.602 |                    6.2 |
| per-query oracle (ceiling)     |   0.66  |                    9.5 |

## Signal 4 — Priors, Finally Tested

The per-cell prior check joins the golden labels back to `cell_selection` and asks: does each archetype's a-priori `predicts` route match what retrieval actually rewards? Mid-sweep, this reading carried an explicit caveat — "dense-friendly cheap lanes dominate until the technical/entity lanes land" — because the cheap BEIR-style lanes finished labelling long before the five join-bug lanes and `scirgen-geo-en` did. That caveat no longer applies: every lane is labelled, so this is now a final reading, not a provisional one.

In [18]:
cell_map = selection[["dataset", "query_id", "cell"]].astype({"query_id": str}).drop_duplicates()
df = current.copy()
df["query_id"] = df["query_id"].astype(str)
per_cell = df.drop(columns=[c for c in ("cell", "stage", "route_selected") if c in df]).merge(
    cell_map, on=["dataset", "query_id"], how="left"
)

predicts = {c.name: set(c.predicts) for c in CELLS}
rows = []
for cell, group in per_cell.groupby("cell"):
    differ = group[group["shape"] == "routes_differ"]
    dist = differ["route"].value_counts()
    top = dist.index[0] if len(dist) else None
    rows.append(
        {
            "cell": cell or "(control)",
            "labelled": len(group),
            "dense": int((differ["route"] == "dense_only").sum()),
            "rrf": int((differ["route"] == "pure_rrf").sum()),
            "sparse": int((differ["route"] == "sparse_only").sum()),
            "measured": top,
            "predicted": ",".join(sorted(predicts.get(cell, ()))) or "-",
            "holds": (top in predicts.get(cell, set())) if top else None,
        }
    )
per_cell_tbl = pd.DataFrame(rows).sort_values("labelled", ascending=False)
show(per_cell_tbl)

held = per_cell_tbl["holds"].dropna()
print(f"predicted route matches measured top route in {int(held.sum())}/{len(held)} cells")

| cell                             |   labelled |   dense |   rrf |   sparse | measured    | predicted              | holds   |
|:---------------------------------|-----------:|--------:|------:|---------:|:------------|:-----------------------|:--------|
| stopword_saturated_midlength     |       8927 |    3349 |   343 |     1104 | dense_only  | dense_only,sparse_only | True    |
| (control)                        |       8141 |    2284 |   156 |     1059 | dense_only  | -                      | False   |
| high_morphological_variation     |       7487 |    2113 |   173 |     1045 | dense_only  | dense_only,sparse_only | True    |
| multi_statement_context_dump     |       5100 |    1314 |   196 |     1439 | sparse_only | dense_only,pure_rrf    | False   |
| negation_bearing_question        |       3668 |    1273 |   109 |      474 | dense_only  | dense_only,sparse_only | True    |
| short_grammatical_question       |       2856 |     983 |    87 |      324 | dense_only  | dense_only,sparse_only | True    |
| wide_flat_enumeration            |       2798 |     575 |   121 |      846 | sparse_only | pure_rrf,sparse_only   | True    |
| extreme_length_pasted_query      |       2014 |     535 |   172 |      761 | sparse_only | dense_only,sparse_only | True    |
| comparative_multi_entity         |       1734 |     377 |    65 |      474 | sparse_only | dense_only,pure_rrf    | False   |
| relative_temporal_no_dates       |       1344 |     405 |    47 |      349 | dense_only  | dense_only             | True    |
| number_inside_natural_question   |       1283 |     391 |    41 |      191 | dense_only  | dense_only,sparse_only | True    |
| deep_nesting_single_sentence     |        962 |     251 |    45 |      144 | dense_only  | dense_only             | True    |
| verbose_grammatical_request      |        752 |     266 |    19 |      159 | dense_only  | dense_only,pure_rrf    | True    |
| keyword_telegram_short           |        683 |     223 |    44 |      127 | dense_only  | dense_only,sparse_only | True    |
| conversational_courtesy_wrapper  |        634 |     197 |    14 |      114 | dense_only  | dense_only             | True    |
| capsword_shape_ambiguity         |        369 |     135 |    30 |       47 | dense_only  | dense_only,sparse_only | True    |
| bare_number_token                |        318 |      91 |    10 |       80 | dense_only  | dense_only,sparse_only | True    |
| web_locator_token                |        316 |     103 |     5 |       66 | dense_only  | dense_only,sparse_only | True    |
| acronym_inside_question          |        315 |      91 |    29 |       52 | dense_only  | pure_rrf,sparse_only   | False   |
| bare_concept_token               |        309 |     114 |    30 |       47 | dense_only  | dense_only             | True    |
| bare_machine_token               |        304 |     106 |     7 |       43 | dense_only  | sparse_only            | False   |
| code_symbol_named_in_prose       |        265 |      77 |     8 |       19 | dense_only  | pure_rrf,sparse_only   | False   |
| version_pinned_technical         |        211 |      68 |     4 |       42 | dense_only  | pure_rrf,sparse_only   | False   |
| instance_value_in_intent         |        176 |      60 |     2 |       61 | sparse_only | dense_only             | False   |
| status_code_idf_split            |        161 |      22 |     2 |       17 | dense_only  | dense_only,sparse_only | True    |
| package_coordinate_dependency    |        151 |      42 |    20 |       60 | sparse_only | pure_rrf,sparse_only   | True    |
| registry_structured_identifier   |        146 |      38 |     1 |       14 | dense_only  | sparse_only            | False   |
| short_quantified_spec            |        129 |      51 |    10 |       19 | dense_only  | dense_only,sparse_only | True    |
| math_notation_present            |        105 |      58 |     0 |       14 | dense_only  | dense_only,sparse_only | True    |
| geo_coordinate_postal            |        103 |      13 |     1 |       29 | sparse_only | dense_only,sparse_only | True    |
| env_var_configuration            |         48 |      12 |     3 |       21 | sparse_only | pure_rrf,sparse_only   | True    |
| rare_key_buried_in_chatter       |         44 |       5 |     1 |        7 | sparse_only | pure_rrf,sparse_only   | True    |
| pasted_code_fragment             |         33 |      13 |     3 |       14 | sparse_only | pure_rrf,sparse_only   | True    |
| logistics_catalog_token          |         27 |       8 |     0 |        3 | dense_only  | sparse_only            | False   |
| bare_acronym                     |         24 |       6 |     4 |        4 | dense_only  | dense_only,sparse_only | True    |
| standards_compliance_lookup      |          6 |       1 |     0 |        1 | sparse_only | pure_rrf,sparse_only   | True    |
| boolean_operator_query           |          6 |       1 |     1 |        2 | sparse_only | dense_only,sparse_only | True    |
| symbol_pile_no_grammar           |          2 |       0 |     0 |        2 | sparse_only | sparse_only            | True    |
| travel_transport_code            |          2 |       0 |     0 |        0 | nan         | dense_only,sparse_only |         |
| bibliographic_catalog_identifier |          2 |       1 |     0 |        1 | sparse_only | dense_only,sparse_only | True    |
| business_temporal_reference      |          2 |       0 |     0 |        0 | nan         | dense_only,sparse_only |         |
| bio_clinical_identifier          |          2 |       0 |     0 |        1 | sparse_only | dense_only,sparse_only | True    |
| single_token_char_blob           |          1 |       0 |     0 |        0 | nan         | sparse_only            |         |
| datetime_token_present           |          1 |       0 |     0 |        0 | nan         | dense_only,sparse_only |         |

predicted route matches measured top route in 30/40 cells


## Reading these four together

None of the four is independent proof on its own — a headroom shift alone could be a fluke of one big lane, a balanced split alone could be coincidence. Together they triangulate: volume grew without diluting quality (1), the resulting balance survives the dataset's own strictest filter (2), that balance is large enough to change which blind policy wins in production terms (3), and the archetype-level priors that predicted all of this hold up now that every lane has actually been measured (4). That's a dataset behaving like the real routing problem it was built to describe, not like an artifact of whichever lane happened to finish labelling first.